In [1]:
import pandas as pd
import xgboost as xgb
import pickle

In [2]:
# 1. 예측할 데이터 불러오기
df_test = pd.read_csv("C:/Users/이민정/Desktop/부트캠프/파이널프로젝트/학습/최종test컬럼.csv")
IDs = df_test['ID']
X_pred = df_test.drop(columns=['ID'])

In [3]:
# 2. 학습된 모델 불러오기
model_e = xgb.XGBClassifier()
model_e.load_model("model/best_model_classification_e_vs_not_e_26.dat")

model_abcd = xgb.XGBClassifier()
model_abcd.load_model("model/best_model_classification_up_downsampling_26.dat")

final_model = xgb.XGBClassifier()
final_model.load_model("model/final_model_2hi.dat")

# 3. 라벨 인코더 불러오기
with open("model/label_encoder_2hi.pkl", "rb") as f:
    le = pickle.load(f)

In [4]:
# 4. 1단계 E vs NOT E 예측
pred_E = model_e.predict(X_pred)
proba_E = model_e.predict_proba(X_pred)[:, 1]

# 5. 2단계 A/B/C/D 예측 (Not E 샘플만)
not_e_idx = pred_E == 0
X_not_e = X_pred.loc[not_e_idx]

pred_ABCD = model_abcd.predict(X_not_e)
proba_ABCD = model_abcd.predict_proba(X_not_e)

In [5]:
# 예측 결과 합치기
df_meta = X_pred.copy()
df_meta['pred_E'] = pred_E
df_meta['proba_E'] = proba_E

# Not E 샘플에만 2단계 결과 입력
df_meta.loc[not_e_idx, 'pred_ABCD'] = pred_ABCD
df_meta.loc[not_e_idx, ['proba_A', 'proba_B', 'proba_C', 'proba_D']] = proba_ABCD

In [6]:
# E 샘플에 pred_ABCD 컬럼에 4 (E 클래스 인덱스) 할당
df_meta.loc[~not_e_idx, 'pred_ABCD'] = 4

# E 샘플에는 A~D 확률을 0으로 채움
df_meta.loc[~not_e_idx, ['proba_A', 'proba_B', 'proba_C', 'proba_D']] = 0

df_meta.fillna(0, inplace=True)

In [7]:
feature_cols = [
    'C_PC1', 'C_PC2', 'C_PC3', '이용금액_R3M_신용체크', '_2순위카드이용금액', '_1순위업종_이용금액',
    '정상입금원금_B5M', '이용금액_오프라인_B0M', '_2순위업종_이용금액', '최대이용금액_일시불_R12M',
    '연체입금원금_B0M', '_3순위쇼핑업종_이용금액', '이용건수_오프라인_R6M', '정상입금원금_B2M',
    '_3순위업종_이용금액', '_1순위교통업종_이용금액', '이용건수_신용_R12M', '쇼핑_도소매_이용금액',
    '이용금액_오프라인_R6M', '청구금액_B0', '청구금액_R6M', '평잔_일시불_3M', '잔액_일시불_B0M',
    'PC1', 'PC2', 'PC3', 'pred_E', 'proba_E', 'pred_ABCD', 'proba_A', 'proba_B', 'proba_C', 'proba_D'
]

# 최종 입력 데이터 생성
X_final_pred = df_meta[feature_cols]

# 8. 최종 메타모델로 최종 예측
final_pred = final_model.predict(X_final_pred)
final_pred_labels = le.inverse_transform(final_pred)

# 9. ID와 최종 예측 결과 합치기
result_df = pd.DataFrame({'ID': IDs, 'Segment': final_pred_labels})

In [8]:
# 10. 결과 저장
result_df.to_csv("2단계계층구조예측결과_1.csv", index=False)
print("저장완료")

저장완료
